In [ ]:
# In[1]:

import pandas as pd

# Load the metric_container.csv file for March 10, 2021
file_path = 'dataset/Bank/telemetry/2021_03_10/metric/metric_container.csv'
metric_container_df = pd.read_csv(file_path)

# Extract unique KPI names and their corresponding components (cmdb_id)
kpi_component_mapping = metric_container_df[['kpi_name', 'cmdb_id']].drop_duplicates().reset_index(drop=True)
kpi_component_mapping

```
Out[1]:
```
The `metric_container.csv` file for March 10, 2021, contains 1,701 unique combinations of KPI names and their corresponding components (cmdb_id). These include KPIs related to CPU utilization, Tomcat request metrics, and container-specific metrics, among others. The components include various Tomcat instances (e.g., Tomcat03, Tomcat04) and Docker containers (e.g., dockerB2). For a detailed analysis, specific KPIs and components can be filtered as needed.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name   cmdb_id
0                            OSLinux-CPU_CPU_CPUCpuUtil  Tomcat04
1     Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  Tomcat03
2     Tomcat-Requests_7441-"http-nio-8003"_Processin...  Tomcat03
3     Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  Tomcat03
4     Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  Tomcat03
...                                                 ...       ...
1696  Container-DOCKER_CONTAINER_d27123361435--bcou-...  dockerB2
1697  Container-DOCKER_CONTAINER_d27123361435--bcou-...  dockerB2
1698  Container-DOCKER_CONTAINER_d27123361435--bcou-...  dockerB2
1699  Container-DOCKER_CONTAINER_d27123361435--bcou-...  dockerB2
1700  Container-DOCKER_CONTAINER_d27123361435--bcou-...  dockerB2

[1701 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Define the possible root cause components
root_cause_components = [
    'apache01', 'apache02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04',
    'MG01', 'MG02', 'IG01', 'IG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02'
]

# Filter the data to include only the KPIs and components matching the root cause components
filtered_metric_container_df = metric_container_df[metric_container_df['cmdb_id'].isin(root_cause_components)]

# Create a new column to classify data by 'component-KPI'
filtered_metric_container_df['component-KPI'] = filtered_metric_container_df['cmdb_id'] + '-' + filtered_metric_container_df['kpi_name']

# Calculate the global P95 threshold for each 'component-KPI' time series
p95_thresholds = filtered_metric_container_df.groupby('component-KPI')['value'].quantile(0.95).reset_index()
p95_thresholds.rename(columns={'value': 'P95_threshold'}, inplace=True)

p95_thresholds

```
Out[2]:
```
The filtered data from `metric_container.csv` includes 1,651 unique 'component-KPI' combinations that match the possible root cause components. For each 'component-KPI' time series, the global P95 threshold was calculated using the entire day's data. These thresholds represent the 95th percentile value for each KPI across the respective components, providing a benchmark for identifying abnormal behavior.

The original code execution output of IPython Kernel is also provided below for reference:

component-KPI  P95_threshold
0         IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryMax   2.040136e+09
1       IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryUsage   4.186372e+01
2        IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryUsed   8.511027e+08
3      IG01-JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed   5.793089e+07
4        IG01-JVM-Operating System_7778_JVM_JVM_CPULoad   5.267200e-01
...                                                 ...            ...
1646              apache02-OSLinux-OSLinux_SWAP_SWAP_Si   0.000000e+00
1647              apache02-OSLinux-OSLinux_SWAP_SWAP_So   0.000000e+00
1648  apache02-OSLinux-OSLinux_SYSTEM_SYSTEM_Check-D...   1.000000e+00
1649  apache02-OSLinux-OSLinux_SYSTEM_SYSTEM_Check-H...   1.000000e+00
1650        apache02-OSLinux-OSLinux_ZABBIX_Host_Uptime   6.586319e+06

[1651 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Define the UTC+8 timezone
timezone = pytz.timezone('Asia/Shanghai')

# Convert the timestamp column to datetime for filtering
filtered_metric_container_df['timestamp'] = pd.to_datetime(filtered_metric_container_df['timestamp'], unit='s').dt.tz_localize('UTC').dt.tz_convert(timezone)

# Filter the data for the time duration between 07:00 and 07:30 on March 10, 2021
start_time = timezone.localize(datetime(2021, 3, 10, 7, 0, 0))
end_time = timezone.localize(datetime(2021, 3, 10, 7, 30, 0))
time_filtered_data = filtered_metric_container_df[(filtered_metric_container_df['timestamp'] >= start_time) & (filtered_metric_container_df['timestamp'] <= end_time)]

# Merge the time-filtered data with the P95 thresholds to identify anomalies
anomaly_data = time_filtered_data.merge(p95_thresholds, on='component-KPI')
anomaly_data['is_anomalous'] = anomaly_data['value'] > anomaly_data['P95_threshold']

# Filter only the anomalous data points
anomalies = anomaly_data[anomaly_data['is_anomalous']].reset_index(drop=True)
anomalies

```
Out[3]:
```
Between 07:00 and 07:30 on March 10, 2021, 199 anomalous data points were identified across various 'component-KPI' time series. These anomalies were detected by comparing the data points against their respective global P95 thresholds. The anomalies span multiple components, including Mysql02, Tomcat02, Mysql01, and MG01, and involve KPIs such as memory usage, CPU wait time, and disk operations. These anomalies indicate potential performance issues during this time period.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                                           kpi_name    value                                      component-KPI  P95_threshold  is_anomalous
0   2021-03-10 07:09:00+08:00   Mysql02       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  89.5067  Mysql02-OSLinux-OSLinux_MEMORY_MEMORY_NoCacheM...      89.492800          True
1   2021-03-10 07:23:00+08:00   Mysql02       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  89.5067  Mysql02-OSLinux-OSLinux_MEMORY_MEMORY_NoCacheM...      89.492800          True
2   2021-03-10 07:29:00+08:00   Mysql02       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  89.5067  Mysql02-OSLinux-OSLinux_MEMORY_MEMORY_NoCacheM...      89.492800          True
3   2021-03-10 07:01:00+08:00  Tomcat02                             OSLinux-CPU_CPU_CPUWio   0.5236                    Tomcat02-OSLinux-CPU_CPU_CPUWio       0.486180          True
4   2021-03-10 07:30:00+08:00   Mysql01      Mysql-MySQL_3306_Innodb os log pending fsyncs   1.0000  Mysql01-Mysql-MySQL_3306_Innodb os log pending...       0.000000          True
..                        ...       ...                                                ...      ...                                                ...            ...           ...
194 2021-03-10 07:16:00+08:00      MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...   0.6000  MG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_D...       0.300000          True
195 2021-03-10 07:28:00+08:00      MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps   0.2000  MG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...       0.183300          True
196 2021-03-10 07:28:00+08:00      MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead   8.0000  MG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...       6.000000          True
197 2021-03-10 07:14:00+08:00      MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps   1.1667  MG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...       1.116700          True
198 2021-03-10 07:14:00+08:00      MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite  16.8000  MG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...      14.873315          True

[199 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Group consecutive anomalies for each 'component-KPI' time series
anomalies['time_diff'] = anomalies.groupby('component-KPI')['timestamp'].diff().dt.total_seconds()
anomalies['fault_group'] = (anomalies['time_diff'] > 60).cumsum()  # Group anomalies with gaps > 60 seconds

# Filter out isolated noise spikes and calculate fault statistics
faults = anomalies.groupby(['component-KPI', 'fault_group']).agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max'),
    max_value=('value', 'max'),
    threshold=('P95_threshold', 'first')
).reset_index()

# Exclude faults where the maximum value only slightly exceeds the threshold (threshold breach <= 50% of the extremal)
faults['threshold_breach'] = (faults['max_value'] - faults['threshold']) / faults['max_value']
filtered_faults = faults[faults['threshold_breach'] > 0.5].reset_index(drop=True)

filtered_faults

```
Out[4]:
```
The analysis identified multiple faults across various 'component-KPI' time series during the time window of 07:00 to 07:30 on March 10, 2021. These faults were grouped based on consecutive anomalies and filtered to exclude isolated noise spikes and minor threshold breaches. The resulting faults show significant threshold breaches (exceeding 50% of the extremal value) and span components such as IG01, IG02, Mysql01, Mysql02, Tomcat01, Tomcat02, apache01, and apache02. These faults indicate potential performance issues that require further investigation.

The original code execution output of IPython Kernel is also provided below for reference:

component-KPI  fault_group                start_time                  end_time  max_value  threshold  threshold_breach
0   IG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...           45 2021-03-10 07:19:00+08:00 2021-03-10 07:19:00+08:00     0.8167   0.200000          0.755112
1   IG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...           45 2021-03-10 07:19:00+08:00 2021-03-10 07:19:00+08:00    47.7333   6.133325          0.871508
2   IG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...           43 2021-03-10 07:20:00+08:00 2021-03-10 07:20:00+08:00    66.0000  20.000000          0.696970
3   IG02-OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE...           19 2021-03-10 07:05:00+08:00 2021-03-10 07:05:00+08:00     1.0000   0.000000          1.000000
4                 Mysql01-Mysql-MySQL_3306_Com Insert            3 2021-03-10 07:00:00+08:00 2021-03-10 07:00:00+08:00     0.2712   0.000000          1.000000
5                 Mysql01-Mysql-MySQL_3306_Com Insert            4 2021-03-10 07:30:00+08:00 2021-03-10 07:30:00+08:00     0.2623   0.000000          1.000000
6   Mysql01-Mysql-MySQL_3306_Innodb data pending f...            4 2021-03-10 07:30:00+08:00 2021-03-10 07:30:00+08:00     1.0000   0.000000          1.000000
7   Mysql01-Mysql-MySQL_3306_Innodb os log pending...            2 2021-03-10 07:30:00+08:00 2021-03-10 07:30:00+08:00     1.0000   0.000000          1.000000
8   Mysql01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sd...           45 2021-03-10 07:05:00+08:00 2021-03-10 07:05:00+08:00    68.8000  18.000000          0.738372
9   Mysql01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sd...           47 2021-03-10 07:26:00+08:00 2021-03-10 07:26:00+08:00    36.8000  18.000000          0.510870
10  Mysql02-Mysql-MySQL_3306_CurrentSQLMaxRunningTime            4 2021-03-10 07:26:00+08:00 2021-03-10 07:26:00+08:00     1.0000   0.000000          1.000000
11         Mysql02-Mysql-MySQL_3306_Handler Read Next            4 2021-03-10 07:26:00+08:00 2021-03-10 07:27:00+08:00    14.9153   0.476680          0.968041
12         Mysql02-Mysql-MySQL_3306_Handler Read Next            5 2021-03-10 07:29:00+08:00 2021-03-10 07:30:00+08:00     9.5333   0.476680          0.949998
13          Mysql02-Mysql-MySQL_3306_Handler Read Rnd            5 2021-03-10 07:26:00+08:00 2021-03-10 07:27:00+08:00     0.7627   0.066700          0.912548
14          Mysql02-Mysql-MySQL_3306_Handler Read Rnd            6 2021-03-10 07:29:00+08:00 2021-03-10 07:30:00+08:00     0.2167   0.066700          0.692201
15      Mysql02-Mysql-MySQL_3306_LongestTrxActiveTime           18 2021-03-10 07:26:00+08:00 2021-03-10 07:26:00+08:00     1.0000   0.000000          1.000000
16             Mysql02-Mysql-MySQL_3306_Opened Tables           17 2021-03-10 07:26:00+08:00 2021-03-10 07:27:00+08:00     0.8033   0.033300          0.958546
17             Mysql02-Mysql-MySQL_3306_Opened Tables           18 2021-03-10 07:29:00+08:00 2021-03-10 07:29:00+08:00     0.0667   0.033300          0.500750
18  Mysql02-Mysql-MySQL_3306_Opened table definitions           17 2021-03-10 07:26:00+08:00 2021-03-10 07:26:00+08:00     0.1639   0.000000          1.000000
19                Mysql02-Mysql-MySQL_3306_Sort Range           13 2021-03-10 07:26:00+08:00 2021-03-10 07:27:00+08:00     0.7627   0.067140          0.911971
20                Mysql02-Mysql-MySQL_3306_Sort Range           14 2021-03-10 07:29:00+08:00 2021-03-10 07:30:00+08:00     0.2167   0.067140          0.690171
21   Mysql02-Mysql-MySQL_3306_Table open cache misses           14 2021-03-10 07:26:00+08:00 2021-03-10 07:27:00+08:00     0.8033   0.033300          0.958546
22   Mysql02-Mysql-MySQL_3306_Table open cache misses           15 2021-03-10 07:29:00+08:00 2021-03-10 07:29:00+08:00     0.0667   0.033300          0.500750
23  Mysql02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sd...           15 2021-03-10 07:19:00+08:00 2021-03-10 07:19:00+08:00     0.8000   0.320000          0.600000
24  Mysql02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sd...           51 2021-03-10 07:01:00+08:00 2021-03-10 07:01:00+08:00     5.0000   1.000000          0.800000
25  Mysql02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sd...           52 2021-03-10 07:03:00+08:00 2021-03-10 07:03:00+08:00     5.0000   1.000000          0.800000
26  Mysql02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sd...           52 2021-03-10 07:05:00+08:00 2021-03-10 07:05:00+08:00     8.0000   0.000000          1.000000
27  Mysql02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sd...           15 2021-03-10 07:25:00+08:00 2021-03-10 07:25:00+08:00     0.2000   0.050000          0.750000
28  Mysql02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sd...           15 2021-03-10 07:25:00+08:00 2021-03-10 07:25:00+08:00    22.4000   1.973333          0.911905
29  Mysql02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sd...           60 2021-03-10 07:27:00+08:00 2021-03-10 07:27:00+08:00    26.0000   9.000000          0.653846
30  Tomcat01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...           37 2021-03-10 07:30:00+08:00 2021-03-10 07:30:00+08:00    11.0000   2.000000          0.818182
31  Tomcat02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...           51 2021-03-10 07:23:00+08:00 2021-03-10 07:23:00+08:00    55.0000  15.000000          0.727273
32           apache01-OSLinux-CPU_CPU-2_SingleCpuUtil           59 2021-03-10 07:14:00+08:00 2021-03-10 07:14:00+08:00    99.6848  44.438480          0.554210
33  apache02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...           28 2021-03-10 07:18:00+08:00 2021-03-10 07:18:00+08:00     5.0000   1.000000          0.800000
34  apache02-OSLinux-OSLinux_NETWORK_NETWORK_TCP-F...           20 2021-03-10 07:19:00+08:00 2021-03-10 07:19:00+08:00     9.0000   0.000000          1.000000```
```

In [ ]:
# In[5]:

# Identify the fault with the most significant deviation from the threshold
most_significant_fault = filtered_faults.loc[filtered_faults['threshold_breach'].idxmax()]

# Determine the fault level based on the component-KPI
fault_component = most_significant_fault['component-KPI']
if 'OSLinux' in fault_component:
    fault_level = 'node-level'
elif 'Container' in fault_component:
    fault_level = 'container-level'
else:
    fault_level = 'service-level'

fault_level, most_significant_fault

```
Out[5]:
```
The fault with the most significant deviation from the threshold occurred at the node level, specifically in the 'IG02-OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE' KPI. This fault was detected at 07:05 on March 10, 2021, with a maximum value of 1.0, completely breaching the threshold of 0.0 (100% deviation). This indicates that the root cause is likely related to node-level network issues on IG02.

The original code execution output of IPython Kernel is also provided below for reference:

('node-level', component-KPI       IG02-OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE...
fault_group                                                        19
start_time                                  2021-03-10 07:05:00+08:00
end_time                                    2021-03-10 07:05:00+08:00
max_value                                                         1.0
threshold                                                         0.0
threshold_breach                                                  1.0
Name: 3, dtype: object)```
```